In [ ]:

# ============================================================
# AMAZON PRODUCT RATING PREDICTION USING DEEP LEARNING
# Dataset: amazon.csv
# Framework: TensorFlow / Keras
# Model: Bidirectional LSTM + Product Features
# ============================================================

import re
import html
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

# 1. REPRODUCIBILITY
tf.keras.utils.set_random_seed(42)
np.random.seed(42)

print("=" * 60)
print("AMAZON PRODUCT RATING PREDICTION")
print("=" * 60)
print("TensorFlow version:", tf.__version__)

# 2. UPLOAD AND LOAD DATASET
uploaded = files.upload()

filename = next(iter(uploaded))
df = pd.read_csv(filename, encoding="utf-8")

# Standardize column names
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

print("\nOriginal dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFirst five records:")
display(df.head())

# 3. CHECK REQUIRED COLUMNS
if "rating" not in df.columns:
    raise ValueError(
        "The dataset must contain a 'rating' column. "
        f"Found columns: {df.columns.tolist()}"
    )

# Create optional columns if absent
text_cols = [
    "product_name", "category", "about_product",
    "review_title", "review_content"
]

for col in text_cols:
    if col not in df.columns:
        df[col] = ""

numeric_cols = [
    "discounted_price",
    "actual_price",
    "discount_percentage"
]

for col in numeric_cols:
    if col not in df.columns:
        df[col] = np.nan

# 4. DATA CLEANING
print("\nMissing values before cleaning:")
print(df.isnull().sum())

df = df.drop_duplicates().copy()

# Convert target rating to numeric
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

# Remove missing or invalid target ratings
df = df.dropna(subset=["rating"]).copy()
df = df[df["rating"].between(1, 5)].copy()

# Convert numeric columns, handling currency symbols and commas
for col in numeric_cols:
    df[col] = (
        df[col].astype(str)
        .str.replace(r"[₹$,\s]", "", regex=True)
        .str.replace("%", "", regex=False)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)

# Clean text columns
for col in text_cols:
    df[col] = df[col].fillna("").astype(str).str.strip()

df["category"] = df["category"].replace("", "Unknown")

print("\nCleaned dataset shape:", df.shape)
print("\nRating distribution:")
print(df["rating"].value_counts().sort_index())

# 5. TEXT PREPROCESSING
def clean_text(text):
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()

df["review_text"] = (
    df["review_title"] + " " +
    df["review_content"] + " " +
    df["product_name"] + " " +
    df["about_product"]
).apply(clean_text)

# Remove records with no usable text
df = df[df["review_text"].str.len() > 0].copy()

# 6. EXPLORATORY DATA ANALYSIS
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="rating", color="steelblue")
plt.title("Amazon Product Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Products")
plt.tight_layout()
plt.show()

print("\nRating statistics:")
print(df["rating"].describe())

# 7. PREPARE INPUTS AND TARGET
X_text = df["review_text"].to_numpy(dtype=str)
X_category = df["category"].to_numpy(dtype=str)
X_num = df[numeric_cols].to_numpy(dtype="float32")
y = df["rating"].to_numpy(dtype="float32")

# Split into 70% training, 15% validation, 15% testing
all_idx = np.arange(len(df))

train_val_idx, test_idx = train_test_split(
    all_idx,
    test_size=0.15,
    random_state=42
)

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.15 / 0.85,
    random_state=42
)

text_train, text_val, text_test = (
    X_text[train_idx], X_text[val_idx], X_text[test_idx]
)

cat_train, cat_val, cat_test = (
    X_category[train_idx], X_category[val_idx], X_category[test_idx]
)

num_train_raw = X_num[train_idx]
num_val_raw = X_num[val_idx]
num_test_raw = X_num[test_idx]

y_train = y[train_idx]
y_val = y[val_idx]
y_test = y[test_idx]

print("\nTraining records:", len(train_idx))
print("Validation records:", len(val_idx))
print("Testing records:", len(test_idx))

# 8. MISSING NUMERIC VALUES AND NORMALIZATION
medians = np.nanmedian(num_train_raw, axis=0)
medians = np.where(np.isfinite(medians), medians, 0.0)

def impute_numeric(data):
    return np.where(
        np.isfinite(data), data, medians
    ).astype("float32")

num_train_clean = impute_numeric(num_train_raw)
num_val_clean = impute_numeric(num_val_raw)
num_test_clean = impute_numeric(num_test_raw)

scaler = StandardScaler()
num_train = scaler.fit_transform(num_train_clean).astype("float32")
num_val = scaler.transform(num_val_clean).astype("float32")
num_test = scaler.transform(num_test_clean).astype("float32")

# 9. TEXT TOKENIZATION AND ENCODING
MAX_TOKENS = 10000
SEQUENCE_LENGTH = 150

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation"
)

vectorizer.adapt(
    tf.data.Dataset.from_tensor_slices(text_train).batch(32)
)

vocab_size = len(vectorizer.get_vocabulary())
print("\nVocabulary size:", vocab_size)

# 10. CATEGORY ENCODING
category_lookup = tf.keras.layers.StringLookup(
    output_mode="one_hot",
    num_oov_indices=1
)

category_lookup.adapt(
    tf.data.Dataset.from_tensor_slices(cat_train).batch(32)
)

print("Category vocabulary size:",
      len(category_lookup.get_vocabulary()))

# 11. BUILD THE MULTI-INPUT BIDIRECTIONAL LSTM MODEL

# Text input
text_input = tf.keras.Input(
    shape=(), dtype=tf.string, name="review"
)

tokens = vectorizer(text_input)

x1 = tf.keras.layers.Embedding(
    input_dim=vocab_size,
    output_dim=64,
    mask_zero=True
)(tokens)

x1 = tf.keras.layers.Bidirectional(
    tf.keras.layers.LSTM(32)
)(x1)

x1 = tf.keras.layers.Dropout(0.3)(x1)

# Numeric product features
numeric_input = tf.keras.Input(
    shape=(len(numeric_cols),),
    dtype=tf.float32,
    name="numeric"
)

x2 = tf.keras.layers.Dense(16, activation="relu")(numeric_input)

# Product category
category_input = tf.keras.Input(
    shape=(), dtype=tf.string, name="category"
)

x3 = category_lookup(category_input)

# Combine all feature branches
combined = tf.keras.layers.Concatenate()([x1, x2, x3])
combined = tf.keras.layers.Dense(64, activation="relu")(combined)
combined = tf.keras.layers.Dropout(0.3)(combined)
combined = tf.keras.layers.Dense(32, activation="relu")(combined)

# Regression output
output = tf.keras.layers.Dense(
    1, activation="linear", name="predicted_rating"
)(combined)

model = tf.keras.Model(
    inputs={
        "review": text_input,
        "numeric": numeric_input,
        "category": category_input
    },
    outputs=output
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

print("\nMODEL SUMMARY")
model.summary()

# 12. PREPARE MODEL INPUTS
X_train = {
    "review": text_train,
    "numeric": num_train,
    "category": cat_train
}

X_val = {
    "review": text_val,
    "numeric": num_val,
    "category": cat_val
}

X_test = {
    "review": text_test,
    "numeric": num_test,
    "category": cat_test
}

# 13. TRAIN MODEL
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

print("\nTRAINING MODEL...")

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

# 14. PLOT TRAINING AND VALIDATION PERFORMANCE
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["mae"], label="Training MAE")
plt.plot(history.history["val_mae"], label="Validation MAE")
plt.title("Training vs Validation MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()

plt.tight_layout()
plt.show()

# 15. TEST SET EVALUATION
y_pred = model.predict(X_test, verbose=0).flatten()
y_pred = np.clip(y_pred, 1, 5)

y_actual_stars = np.rint(y_test).astype(int)
y_pred_stars = np.clip(np.rint(y_pred), 1, 5).astype(int)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
accuracy = accuracy_score(y_actual_stars, y_pred_stars)

print("\n" + "=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)
print(f"MAE: {mae:.4f} stars")
print(f"RMSE: {rmse:.4f} stars")
print(f"R2 Score: {r2:.4f}")
print(f"Rounded-star accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(
    y_actual_stars,
    y_pred_stars,
    labels=[1, 2, 3, 4, 5],
    zero_division=0
))

# 16. ACTUAL VS PREDICTED RATINGS
results = pd.DataFrame({
    "Actual Rating": y_test,
    "Predicted Rating": y_pred,
    "Predicted Stars": y_pred_stars,
    "Absolute Error": np.abs(y_test - y_pred)
})

print("\nACTUAL VS PREDICTED RATINGS")
display(results.head(20).round(2))

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([1, 5], [1, 5], "r--", label="Perfect prediction")
plt.xlabel("Actual Rating")
plt.ylabel("Predicted Rating")
plt.title("Actual vs Predicted Ratings")
plt.xlim(1, 5)
plt.ylim(1, 5)
plt.legend()
plt.grid(True)
plt.show()

# 17. CONFUSION MATRIX
cm = confusion_matrix(
    y_actual_stars,
    y_pred_stars,
    labels=[1, 2, 3, 4, 5]
)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[1, 2, 3, 4, 5],
    yticklabels=[1, 2, 3, 4, 5]
)
plt.xlabel("Predicted Rating")
plt.ylabel("Actual Rating")
plt.title("Rating Confusion Matrix")
plt.tight_layout()
plt.show()

# 18. BASELINE COMPARISON
baseline_pred = np.full(len(y_test), y_train.mean())

comparison = pd.DataFrame({
    "Model": ["Mean Baseline", "Bidirectional LSTM"],
    "MAE": [
        mean_absolute_error(y_test, baseline_pred),
        mean_absolute_error(y_test, y_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, baseline_pred)),
        np.sqrt(mean_squared_error(y_test, y_pred))
    ]
})

print("\nBASELINE COMPARISON")
display(comparison.round(4))

# 19. FUNCTION FOR UNSEEN REVIEW PREDICTION
def predict_rating(
    review,
    category="Unknown",
    discounted_price=None,
    actual_price=None,
    discount_percentage=None,
    product_name="",
    about_product=""
):
    combined_text = clean_text(
        str(review) + " " +
        str(product_name) + " " +
        str(about_product)
    )

    supplied = {
        "discounted_price": discounted_price,
        "actual_price": actual_price,
        "discount_percentage": discount_percentage
    }

    feature_values = []

    for i, col in enumerate(numeric_cols):
        value = supplied[col]
        if value is None:
            value = medians[i]
        feature_values.append(float(value))

    numeric_array = np.array(
        feature_values, dtype="float32"
    ).reshape(1, -1)

    numeric_scaled = scaler.transform(
        numeric_array
    ).astype("float32")

    prediction = model.predict(
        {
            "review": np.array([combined_text], dtype=str),
            "numeric": numeric_scaled,
            "category": np.array([category], dtype=str)
        },
        verbose=0
    )[0, 0]

    prediction = float(np.clip(prediction, 1, 5))
    stars = int(np.clip(np.rint(prediction), 1, 5))

    print("\nNEW REVIEW PREDICTION")
    print("Review:", review)
    print("Predicted rating:", round(prediction, 2), "/ 5")
    print("Predicted whole-star rating:", stars, "/ 5")

    return prediction

# 20. DEMONSTRATE ON UNSEEN REVIEWS
predict_rating(
    review="Excellent product with great sound quality and battery life.",
    category="Electronics",
    discounted_price=999,
    actual_price=1499,
    discount_percentage=33,
    product_name="Wireless headphones"
)

predict_rating(
    review="Very poor quality. It stopped working after two days.",
    category="Electronics",
    discounted_price=999,
    actual_price=1499,
    discount_percentage=33,
    product_name="Wireless headphones"
)

# 21. SAVE MODEL AND PREPROCESSING OBJECTS
model.save("amazon_rating_model.keras")

joblib.dump(scaler, "amazon_scaler.pkl")
joblib.dump(numeric_cols, "amazon_numeric_columns.pkl")
joblib.dump(medians, "amazon_numeric_medians.pkl")

print("\nModel and preprocessing files saved.")

# 22. DOWNLOAD OUTPUT FILES
files.download("amazon_rating_model.keras")
files.download("amazon_scaler.pkl")
files.download("amazon_numeric_columns.pkl")
files.download("amazon_numeric_medians.pkl")

print("\nPROJECT COMPLETED SUCCESSFULLY!")
print("Use the printed evaluation metrics in your report.")
print("Do not claim a specific accuracy before observing your results.")
